# Agentic AI, Day 5 Capstone: Build the Complete Agent

This is the finale. You will assemble everything from the week into one agent:

- the **ReAct loop** from Day 3 (reason, act, observe),
- **tools** from Day 1 (weather, calculator),
- a **RAG knowledge tool** from Day 4 (look facts up in your docs),
- **memory** from Day 4 (remember across the conversation),
- **guardrails** from today (an allow-list and argument checks),
- and a **human approval gate** from today (sign-off before a risky action).

Nothing here is brand new. It is assembly. We build it up one layer at a time, and there is a single place where tools run, `run_tool`, that we upgrade as we add safety.

Run the cells in order, top to bottom.

## Setup

You need a chat model (`llama3.1:8b` or `qwen2.5:7b`) and the embedding model from Day 4. If you have not pulled it:

```
ollama pull nomic-embed-text
```

The cell below reloads everything from the week: the `ask` helper, the Day 1 tools, the Day 4 embedding and retrieval pieces, and two new tools for today (a knowledge lookup and a pretend email sender). **Change `MODEL` if yours differs.**

In [ ]:
import ollama, ast, operator
import numpy as np

MODEL = "llama3.1"
EMBED_MODEL = "nomic-embed-text"

def ask(prompt: str) -> str:
    """Send one prompt to the chat model and return the reply."""
    return ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}]).message.content

# ---- Day 1 tools ----
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The city name, for example "Pune".
    """
    data = {"pune": "31C, clear sky", "mumbai": "33C, humid",
            "delhi": "29C, hazy", "bangalore": "26C, light rain"}
    return data.get(city.lower(), f"No weather data for {city}")

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def _ev(n):
    if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
    if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_ev(n.left), _ev(n.right))
    if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_ev(n.operand))
    raise ValueError("Unsupported expression")
def calculate(expression: str) -> str:
    """Evaluate a basic arithmetic expression like "33 - 31".

    Args:
        expression: The arithmetic expression to evaluate.
    """
    return str(_ev(ast.parse(expression, mode="eval").body))

# ---- Day 4 retrieval ----
def embed(text):
    return np.array(ollama.embed(model=EMBED_MODEL, input=text).embeddings[0])
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

DOCS = [
    "The Trinity office is open from 9am to 6pm on weekdays.",
    "Employees get 24 days of paid leave per year.",
    "The cafeteria serves lunch between 12pm and 2pm.",
    "Remote work is allowed up to three days per week.",
    "The IT helpdesk can be reached at extension 4500.",
]
DOC_VECS = [embed(d) for d in DOCS]

# ---- new tools for the capstone ----
def knowledge_search(query: str) -> str:
    """Look up facts about the Trinity office: hours, leave, remote work, cafeteria, IT.

    Args:
        query: What to look up, in plain words.
    """
    q = embed(query)
    sims = [cosine(q, v) for v in DOC_VECS]
    order = sorted(range(len(DOCS)), key=lambda i: sims[i], reverse=True)
    return " | ".join(DOCS[i] for i in order[:2])

def send_email(to: str, body: str) -> str:
    """Send an email. Use only when the user explicitly asks to send something.

    Args:
        to: The recipient address.
        body: The message body.
    """
    return f"(pretend) Email sent to {to}. Body: {body[:60]}"

SYSTEM = (
    "You are a helpful assistant for the Trinity office. Think briefly about "
    "what to do, use tools for weather, math, office facts, and sending email, "
    "and stop and answer once you have what you need."
)

print("Loaded. Chat:", MODEL, " Embedding:", EMBED_MODEL, " Docs indexed:", len(DOCS))

## Milestone 1: The core ReAct agent

Start with the Day 3 loop. We define one shared place where tools run, `run_tool`, and the `react` loop that calls it. Right now `run_tool` just runs the tool with basic error handling. We will upgrade it later without touching the loop.

In [ ]:
def run_tool(name, args, tools_map):
    """The single place where tools execute. We will upgrade this later."""
    fn = tools_map.get(name)
    if fn is None:
        return f"Error: no tool named '{name}'."
    try:
        return str(fn(**args))
    except Exception as e:
        return f"Tool error: {e}"

def react(question, tools_list, tools_map, max_steps=8):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": question}]
    for step in range(max_steps):
        res = ollama.chat(model=MODEL, messages=msgs, tools=tools_list)
        msgs.append(res.message)
        if not res.message.tool_calls:
            return res.message.content
        for c in res.message.tool_calls:
            out = run_tool(c.function.name, c.function.arguments or {}, tools_map)
            print(f"  [tool] {c.function.name}({c.function.arguments}) -> {out[:60]}")
            msgs.append({"role": "tool", "content": out})
    return "Stopped: reached the step budget."

basic_list = [get_weather, calculate]
basic_map = {"get_weather": get_weather, "calculate": calculate}
print(react("Is it hotter in Pune or Mumbai, and by how many degrees?", basic_list, basic_map))

**What you should see:** the trace shows `get_weather` twice and `calculate` once, then a final answer. This is your Day 3 agent, rebuilt. Everything else today plugs into this.

## Milestone 2: Add a knowledge tool

The agent only knows the four fake cities. To answer questions about the office, give it the RAG search from Day 4, wrapped as a tool. Because `react` is generic over its tool list, we just pass one more tool. No change to the loop.

In [ ]:
tools_list = [get_weather, calculate, knowledge_search]
tools_map = {"get_weather": get_weather, "calculate": calculate,
             "knowledge_search": knowledge_search}

print(react("How many days of paid leave do employees get?", tools_list, tools_map))

**What you should see:** the agent calls `knowledge_search`, which retrieves the leave fact, then answers 24 days. The agent now reaches into your documents when it needs to, and uses the calculator or weather tool when those fit instead.

## Milestone 3: Add memory

So far each question starts fresh. Now add conversation memory from Day 4: a running summary plus the most recent turns. We define `agent`, which is the same loop seeded with memory at the start and updating it at the end. It still calls the shared `run_tool`.

In [ ]:
def agent(question, state, tools_list, tools_map, max_steps=8, keep_turns=2):
    """ReAct loop, but seeded with conversation memory and updating it afterwards."""
    msgs = [{"role": "system", "content": SYSTEM + " Earlier context: " + state["summary"]}]
    msgs += state["recent"][-(2 * keep_turns):]
    msgs.append({"role": "user", "content": question})

    answer = "Stopped: reached the step budget."
    for step in range(max_steps):
        res = ollama.chat(model=MODEL, messages=msgs, tools=tools_list)
        msgs.append(res.message)
        if not res.message.tool_calls:
            answer = res.message.content
            break
        for c in res.message.tool_calls:
            out = run_tool(c.function.name, c.function.arguments or {}, tools_map)
            print(f"  [tool] {c.function.name}({c.function.arguments}) -> {out[:60]}")
            msgs.append({"role": "tool", "content": out})

    # update conversation memory
    state["recent"].append({"role": "user", "content": question})
    state["recent"].append({"role": "assistant", "content": answer})
    if len(state["recent"]) > 2 * keep_turns:
        old = state["recent"][:-(2 * keep_turns)]
        state["summary"] = ask("Update this summary with key facts, keep it short.\n"
                               f"Old: {state['summary']}\nNew: {old}")
        state["recent"] = state["recent"][-(2 * keep_turns):]
    return answer

state = {"recent": [], "summary": "none yet"}
agent("My name is Priya and I manage the Pune team.", state, tools_list, tools_map)
print("---")
print(agent("What is my name, and what is the weather where my team is?", state, tools_list, tools_map))

**What you should see:** the second answer knows your name is Priya and that your team is in Pune (from memory), then calls `get_weather` for Pune (a tool). Memory and tools working together in one agent.

## Milestone 4: Add guardrails

Right now the agent would run any tool the model names, with any arguments. That is unsafe. We upgrade the one place tools execute, `run_tool`, to add an allow-list and an argument check. The `agent` loop does not change at all, because it always calls `run_tool`.

In [ ]:
ALLOWED = {"get_weather", "calculate", "knowledge_search"}

def run_tool(name, args, tools_map):
    """Upgraded: only allowed tools run, and arguments are checked first."""
    if name not in ALLOWED:
        return f"Blocked: '{name}' is not on the allow-list."
    if name == "calculate" and len(str(args.get("expression", ""))) > 100:
        return "Blocked: expression too long."
    fn = tools_map.get(name)
    if fn is None:
        return f"Error: no tool named '{name}'."
    try:
        return str(fn(**args))
    except Exception as e:
        return f"Tool error: {e}"

# prove the guardrail works, without involving the model
print(run_tool("delete_everything", {}, tools_map))          # blocked: not allowed
print(run_tool("get_weather", {"city": "Pune"}, tools_map))  # allowed

**What you should see:** the first call is blocked because `delete_everything` is not on the allow-list, and the second runs normally. Even if the model hallucinated a dangerous tool name, it could not run. This is the single highest-value guardrail for a tool-using agent.

## Milestone 5: Add a human approval gate

Some actions should not happen without a person. We add `send_email` to the allowed tools, mark it as needing approval, and upgrade `run_tool` once more to pause and ask before running anything risky. Safe tools still run freely.

**Note:** the approval step uses `input`, so it will pause and wait for you to type `y` or `n`.

In [ ]:
ALLOWED = {"get_weather", "calculate", "knowledge_search", "send_email"}
NEEDS_APPROVAL = {"send_email"}

def run_tool(name, args, tools_map):
    """Upgraded again: risky tools require human approval before running."""
    if name not in ALLOWED:
        return f"Blocked: '{name}' is not on the allow-list."
    if name == "calculate" and len(str(args.get("expression", ""))) > 100:
        return "Blocked: expression too long."
    fn = tools_map.get(name)
    if fn is None:
        return f"Error: no tool named '{name}'."

    if name in NEEDS_APPROVAL:
        print(f"\n  [approval needed] the agent wants to run {name}({args})")
        if input("  Approve? (y/n) ").strip().lower() != "y":
            return "Action cancelled by the human."

    try:
        return str(fn(**args))
    except Exception as e:
        return f"Tool error: {e}"

# add the email tool to the agent's toolset
tools_list = [get_weather, calculate, knowledge_search, send_email]
tools_map = {"get_weather": get_weather, "calculate": calculate,
             "knowledge_search": knowledge_search, "send_email": send_email}

state = {"recent": [], "summary": "none yet"}
print(agent("Send a one-line hello to test@trinity.com", state, tools_list, tools_map))

**What you should see:** when the agent tries to send the email, it pauses and asks you to approve. Type `y` and it sends (the pretend version), type `n` and it reports the action was cancelled. Read-only tools like weather still run without asking.

## Milestone 6: Demo and evaluate

First, a full demo that exercises several layers at once: retrieval plus a guarded, approved action. Then we evaluate the agent against a small golden set, so we have a number instead of a vibe.

In [ ]:
# full demo: look something up, then send it (you will be asked to approve)
state = {"recent": [], "summary": "none yet"}
print(agent("Find our leave policy and email a one-line summary to manager@trinity.com",
            state, tools_list, tools_map))

**What you should see:** the agent calls `knowledge_search` for the leave policy, then asks you to approve the `send_email` action. Retrieval, guardrails, and human oversight, all in one run.

In [ ]:
# evaluate against a small golden set (none of these trigger the approval gate)
golden = [
    ("What is the weather in Pune?", "31"),
    ("What is 9 times 8?", "72"),
    ("How many days of leave do employees get?", "24"),
]

passed = 0
for q, expected in golden:
    fresh = {"recent": [], "summary": "none"}
    out = agent(q, fresh, tools_list, tools_map)
    ok = expected in out
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  {q}")

print(f"\nScore: {passed} / {len(golden)}")

**What you should see:** a PASS or FAIL per question and a final score. On a small local model you may not get a perfect score every run, and running it again may give a different result. That variation is exactly why a golden set beats judging from a single example.

**Your turn:** add two more cases to the golden set, including one about remote work, and rerun.

## You built it

This one agent uses the entire week:

- **ReAct** to reason and act in a loop,
- **tools** for weather and math,
- **RAG** to look facts up in your documents,
- **memory** to remember across the conversation,
- **guardrails** to block disallowed tools and bad arguments,
- a **human approval gate** for the one risky action,
- and an **evaluation** to measure whether it works.

That is a real, safe, useful agent, running entirely on your own machine.

### Optional challenges

1. **Make it multi-agent:** add a `checker` agent that reviews the final answer before it is returned, a simple pipeline.
2. **Output guardrail:** before returning, block any answer that contains an email address or a phone number.
3. **Persist memory:** save the conversation summary to a file and load it next time, so memory survives a restart.
4. **Grow the golden set:** write ten cases covering every tool, and track your score as you tweak prompts.

### Where to go next

You went from a single model call to a complete agent in five days. From here:

- read the rest of Gulli's patterns, especially MCP, evaluation, and optimization,
- rebuild this capstone in a framework like LangGraph or CrewAI, and you will recognise every piece,
- and point an agent at a real problem you actually care about.

Congratulations, and now go build.